In [11]:
import yfinance as yf
import pandas as pd

def audit_ticker(ticker_symbol):
    """
    Fetches financial statements via yfinance and flags critical concerns.
    """
    ticker = yf.Ticker(ticker_symbol)
    
    # 1. Fetch Data
    # .financials returns the Income Statement; .balance_sheet returns the Balance Sheet
    income_stmt = ticker.financials
    balance_sheet = ticker.balance_sheet
    
    if income_stmt.empty or balance_sheet.empty:
        return f"Error: Could not retrieve data for {ticker_symbol}."

    # Use the most recent year's data (first column)
    latest_is = income_stmt.iloc[:, 0]
    latest_bs = balance_sheet.iloc[:, 0]

    report = [f"### FINANCIAL AUDIT REPORT: {ticker_symbol.upper()}"]
    flags = []

    # 2. Liquidity Check (Current Ratio)
    ca = latest_bs.get('Current Assets', 0)
    cl = latest_bs.get('Current Liabilities', 0)
    current_ratio = ca / cl if cl > 0 else 0
    
    if current_ratio < 1.0:
        flags.append(f"CRITICAL LIQUIDITY: Current Ratio is {current_ratio:.2f}. The company cannot cover short-term debts with current assets.")
    elif current_ratio < 1.5:
        flags.append(f"LOW LIQUIDITY: Current Ratio is {current_ratio:.2f}. Potential cash flow tightening.")

    # 3. Solvency Check (Debt-to-Equity)
    total_debt = latest_bs.get('Total Debt', 0)
    equity = latest_bs.get('Stockholders Equity', 0)
    de_ratio = total_debt / equity if equity > 0 else 0
    
    if de_ratio > 2.0:
        flags.append(f"HIGH LEVERAGE: Debt-to-Equity is {de_ratio:.2f}. Significant reliance on borrowed capital.")

    # 4. Profitability Check (Net Margin)
    revenue = latest_is.get('Total Revenue', 0)
    net_income = latest_is.get('Net Income', 0)
    margin = (net_income / revenue) * 100 if revenue > 0 else 0
    
    if margin < 0:
        flags.append(f"NEGATIVE PROFITABILITY: The company reported a net loss. Margin: {margin:.1f}%.")
    elif margin < 5:
        flags.append(f"SLIM MARGINS: Net Profit Margin is only {margin:.1f}%, leaving little room for operational errors.")

    # 5. Efficiency Check (Inventory/AR vs Revenue)
    inventory = latest_bs.get('Inventory', 0)
    if inventory > (0.5 * revenue): # Arbitrary threshold for warning
        flags.append(f"INVENTORY CLOG: Inventory levels are unusually high relative to revenue, suggesting slow-moving stock.")

    # Format Output
    if not flags:
        report.append("* No immediate areas of concern detected based on standard benchmarks.")
    else:
        for flag in flags:
            report.append(f"* {flag}")

    return "\n".join(report)

# Example Usage
print(audit_ticker("SCI"))

### FINANCIAL AUDIT REPORT: SCI
* CRITICAL LIQUIDITY: Current Ratio is 0.55. The company cannot cover short-term debts with current assets.
* HIGH LEVERAGE: Debt-to-Equity is 3.18. Significant reliance on borrowed capital.
